# Phase 3: Evidence PDF Generation — Theory & Lab
## Audit-Ready Reports, ReportLab, and Lambda Layers

**Time Estimate:** 6–8 hours | **Prerequisites:** Phase 2 completed (control assessments)

---

### What you'll build
A production PDF report generator that takes `ControlAssessment` objects from Phase 2 and produces audit-ready compliance evidence packages — the same output real FedRAMP auditors review.

### Study cross-references
| Concept | DDIA Chapter | DVA-C02 | System Design Interview |
|---------|-------------|---------|-------------------------|
| PDF as derived dataset | Ch. 10: Batch Processing | Lambda batch operations | Ch. 15: Google Drive |
| Schema evolution in reports | Ch. 4: Encoding and Evolution | API versioning | — |
| Output immutability | Ch. 10: Batch Processing (p. 413) | S3 object immutability | — |
| Partitioned output (by family) | Ch. 6: Partitioning | S3 prefix partitioning | Ch. 5: Consistent Hashing |

### Documentation links
- [ReportLab User Guide (PDF)](https://www.reportlab.com/docs/reportlab-userguide.pdf)
- [ReportLab GitHub](https://github.com/ReportLab/reportlab)
- [Jinja2 Template Docs](https://jinja.palletsprojects.com/)
- [S3 Presigned URLs](https://docs.aws.amazon.com/AmazonS3/latest/userguide/PresignedUrlUploadObject.html)
- [Lambda Layers](https://docs.aws.amazon.com/lambda/latest/dg/chapter-layers.html)
- [FedRAMP SSP Template](https://www.fedramp.gov/documents-templates/)
- [NIST SP 800-53 Rev 5](https://csrc.nist.gov/publications/detail/sp/800-53/rev-5/final)

---

---
## 🔗 How This Phase Connects to Everything You've Built

Before writing a single line, orient yourself. Every notebook in this project chains output from the previous phase as input to the next.

```
Phase 1 (Collectors)          Phase 2 (Mapper)              Phase 3 (PDF) ← YOU ARE HERE
─────────────────────         ────────────────────          ──────────────────────────────
boto3 API calls               for each control_id:          take assessments + posture
  ↓                             gather matching evidence     render to PDF
EvidenceItem[]                  assess status               write to disk / S3
  ↓                             calculate priority
ScanResult                    ControlAssessment[]
  .collector_results[]          CompliancePosture
```

**The data types never change.** `EvidenceItem`, `ScanResult`, `ControlAssessment`, and `CompliancePosture` are defined once in `src/models.py` and flow through all three phases unchanged. The PDF generator doesn't know or care how the data was collected — it just receives a list of `ControlAssessment` objects.

**Why this matters for system design:** This is the *Unix pipe philosophy* — small tools that do one thing and chain together via a shared data contract. DDIA Ch. 10 calls this "derived datasets": each phase produces an output that becomes the immutable input to the next.

**Boilerplate recognition:** Every phase follows the same structure:
```python
# 1. Import from src/  (always the same pattern)
# 2. Build/receive input data
# 3. Process it
# 4. Return/write output
```
The imports change, the types change, but the shape of the code stays identical. This is intentional — recognizing that shape is a senior engineering skill.

---

## Part 1: Theory — What Auditors Actually Expect

### The Audit Chain of Custody

When you submit evidence to an auditor (SOC 2, FedRAMP, HIPAA), they expect:

1. **Timeliness** — Evidence must be dated and timestamped (ISO 8601 UTC)
2. **Completeness** — Every control finding must be included with context
3. **Chain of Custody** — How was this data collected? (source system, collection method)
4. **Authenticity** — Cryptographic hash or digital signature proving it hasn't been altered
5. **Legibility** — Professional formatting, clear structure
6. **Traceable** — Auditor can link each claim back to raw logs/findings

### What a Compliant Evidence Package Looks Like

```
Compliance_Evidence_Report_2026-05-03.pdf
├── Cover Page (org name, date, framework, scan ID)
├── Executive Summary (compliance %, critical findings, status breakdown)
├── Control Family Reports (one section per family)
│   ├── AC (Access Control) — family summary + per-control evidence
│   ├── AU (Audit & Accountability)
│   └── ...
├── Finding Details (sorted by severity, with remediation steps)
└── Remediation Guide (prioritized action items)
```

### Audit Best Practices
- **No Editable PDFs** — Use encrypted, flattened PDFs
- **Include Metadata** — PDF creation date, generator version, author
- **Page Numbers & Headers** — Auditors reference findings by page
- **Font Consistency** — System fonts (Helvetica, Times) avoid embedding issues
- **Color Contrast** — Must be readable in black & white print

---

---
### 📋 Audit Requirements Encoded in Python: Read `src/models.py` as a Compliance Spec

Every field in `EvidenceItem` maps directly to an audit requirement from the section above. This is not a coincidence — it's intentional design.

```python
@dataclass
class EvidenceItem:
    source: str          # Chain of Custody: "Which system produced this evidence?"
                         # e.g., "security_hub", "config", "cloudtrail"

    finding_id: str      # Traceability: unique ID so auditor can look up the raw finding
                         # e.g., "sh-iam4-001" → Security Hub finding in the console

    title: str           # Legibility: human-readable description

    status: str          # Completeness: PASSED / FAILED / NOT_EVALUATED

    severity: str        # Completeness: CRITICAL / HIGH / MEDIUM / LOW / INFORMATIONAL

    resource_type: str   # Traceability: "What AWS resource was checked?"
    resource_id: str     # Traceability: "Which specific resource?"
                         # e.g., "arn:aws:s3:::production-data-bucket"

    timestamp: str       # Timeliness: ISO 8601 UTC — when was this checked?

    remediation: str     # Legibility: what should be done if this failed?

    control_ids: List[str]  # Completeness: which NIST controls does this satisfy?
```

**Why dataclasses?** Three reasons:
1. `__init__` is generated automatically — no boilerplate constructor
2. `__repr__` is generated — `print(evidence_item)` is readable in the notebook
3. Type hints are enforced at the field level, not hidden in docstrings

**The `raw_data` field (not shown above):**  
`EvidenceItem` also has `raw_data: dict = field(default_factory=dict)` — the original API response from Security Hub or Config. This satisfies the "authenticity" requirement: the auditor can request the raw API output to verify the finding independently.

**DDIA Connection (Ch. 4 — Encoding):** `EvidenceItem` is our internal schema. When we serialize it for storage (`to_dict()`) or transmission (JSON to Lambda), we're encoding it. When we load it back (`from_dict()`), we're decoding. Schema evolution (adding optional fields) is backward-compatible as long as old `from_dict()` implementations use `.get()` with defaults.

---

## Part 2: ReportLab Foundations

### Why ReportLab?

| Feature | ReportLab | weasyprint | FPDF | Word→PDF |
|---------|-----------|-----------|------|----------|
| **Python-native** | ✓ | ✓ | ✓ | ✗ |
| **No browser needed** | ✓ | ✗ (needs Chrome) | ✓ | ✗ |
| **Lambda-friendly** | ✓ | ✗ (500MB+) | ✓ | ✗ |
| **Complex layouts** | ✓ | ✓ | — | ✓ |

ReportLab is ideal because it runs in AWS Lambda (small footprint), gives full programmatic control, and produces deterministic output (same input = identical PDF).

### ReportLab Core Concepts

**Three-layer model:**

```
┌─────────────────────────────────────┐
│ High Level (Flowables)              │  ← Paragraphs, Tables, PageBreaks
├─────────────────────────────────────┤
│ Canvas (Low Level Drawing)          │  ← Lines, Rectangles, Text, Images
├─────────────────────────────────────┤
│ PDF Document                        │  ← Binary output
└─────────────────────────────────────┘
```

**Flowables** (what we'll use):
- `Paragraph` — Text with styling
- `Table` — Rows and columns with cell styling
- `PageBreak` — New page
- `Spacer` — Whitespace
- `SimpleDocTemplate` — Builds the PDF from a list of flowables

---

---
### 🧱 Pattern Recognition: The Builder Pattern

Before touching code, recognize the shape of how ReportLab works:

```python
story = []                          # Step 1: create an empty accumulator
story.append(Paragraph(...))        # Step 2: add items one at a time
story.append(Table(...))
story.append(PageBreak())
doc.build(story)                    # Step 3: render everything at once
```

This is the **Builder Pattern** — you collect all the parts first, then build the final product in one shot. You see this pattern constantly across languages and tools:

| Library / Tool | Accumulator | Final Build Call |
|---|---|---|
| ReportLab | `story = []` | `doc.build(story)` |
| AWS CloudFormation | Resources dict | `create_stack()` |
| boto3 paginators | `results = []` — `results.extend(page)` | `return results` |
| SQL query builder | `.filter().order_by()` | `.all()` / `.execute()` |
| pytest | test functions | `pytest` discovers + runs |

**Why not render immediately?** ReportLab needs to know the *total page count* before it can print page numbers. You can't put "Page 1 of 3" on page 1 until you've laid out pages 2 and 3. The builder collects everything first, then does a two-pass render.

**The most common gotcha — `buffer.seek(0)`:**
```python
buffer = BytesIO()
doc.build(story)       # cursor is now at the END of the buffer
buffer.seek(0)         # ← MUST do this before reading
data = buffer.read()   # without seek(0), you'd read 0 bytes
```
Think of BytesIO like a cassette tape: after recording, the tape head is at the end. You must rewind before playing it back.

---

## Lab 3.1: ReportLab Basics

**Goal:** Generate a simple PDF with text, tables, and styled elements. This is standalone ReportLab practice before we wire it up to our `src/` classes.

In [ ]:
# Install ReportLab (skip if already installed)
!pip install reportlab -q

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from io import BytesIO
from datetime import datetime

# Create an in-memory PDF
buffer = BytesIO()
doc = SimpleDocTemplate(buffer, pagesize=letter)
styles = getSampleStyleSheet()

# Custom title style
title_style = ParagraphStyle(
    'CustomTitle', parent=styles['Heading1'],
    fontSize=24, textColor=colors.HexColor('#003366'),
    spaceAfter=30, alignment=1  # Center
)

# Build content as a list of "flowables"
story = []
story.append(Paragraph("Compliance Evidence Report", title_style))
story.append(Spacer(1, 0.2*inch))
story.append(Paragraph(f"<b>Generated:</b> {datetime.utcnow().isoformat()}Z", styles['Normal']))
story.append(Spacer(1, 0.3*inch))

# Summary table
table_data = [
    ['Control Family', 'Total', 'Compliant', 'Non-Compliant'],
    ['AC (Access Control)', '3', '1', '2'],
    ['AU (Audit & Accountability)', '4', '4', '0'],
    ['SC (System & Communications)', '4', '2', '2'],
]

table = Table(table_data, colWidths=[2.5*inch, 1*inch, 1.5*inch, 1.5*inch])
table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#003366')),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('GRID', (0, 0), (-1, -1), 1, colors.grey),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f0f0f0')]),
]))
story.append(table)

doc.build(story)
buffer.seek(0)
with open(r'C:\Users\jkl91\Downloads\hello_world.pdf', 'wb') as f: 
    f.write(buffer.getvalue())

print(f"PDF created: {len(buffer.getvalue()) / 1024:.1f} KB")
print("Key concept: story = list of flowables, doc.build(story) = render to PDF") 


import os                                                                                                                                                                                                         
print(os.getcwd()) 

PDF created: 2.0 KB
Key concept: story = list of flowables, doc.build(story) = render to PDF
c:\Users\jkl91\Documents\jonathanlohr-portfolio\jonathanlohr-portfolio\AWS Compliance collector\notebooks\phase-3-evidence-pdf



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\jkl91\AppData\Local\Temp\ipykernel_27276\3602465644.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  story.append(Paragraph(f"<b>Generated:</b> {datetime.utcnow().isoformat()}Z", styles['Normal']))


---
### 🔬 Line-by-Line: What Just Happened in Lab 3.1

Break down every new concept from that code cell:

**Imports — what each package does:**
```python
from reportlab.lib.pagesizes import letter   # (8.5*inch, 11*inch) tuple — page dimensions
from reportlab.lib.styles import getSampleStyleSheet  # prebuilt CSS-like styles
from reportlab.lib.units import inch         # 1 inch = 72 PDF points
from reportlab.lib import colors             # color constants + HexColor()
from reportlab.platypus import ...           # flowables: the building blocks
from io import BytesIO                       # in-memory file object — no disk write yet
```

**`ParagraphStyle` — CSS inheritance in Python:**
```python
title_style = ParagraphStyle(
    'CustomTitle',              # internal name (must be unique)
    parent=styles['Heading1'],  # inherit all Heading1 defaults, then override below
    fontSize=24,
    textColor=colors.HexColor('#003366'),
    alignment=1                 # 0=left, 1=center, 2=right
)
```
`parent=` is CSS class inheritance. If you don't set a property, it falls through to the parent. This is how `styles['Normal']` and `styles['Heading1']` work — they all inherit from a base `ParagraphStyle`.

**`TableStyle` commands — applied top to bottom, later overrides earlier:**
```python
TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#003366')),
    #              ^start   ^end     ^value
    #    (col, row)         (col, row)
    #    (0,0) = top-left   (-1, 0) = top-right (last col, first row)
    #    (-1, -1) = bottom-right corner
])
```
Think of it like CSS specificity — the last matching command wins. `ROWBACKGROUNDS` is a shortcut that alternates colors across rows automatically.

**`Spacer(1, 0.2*inch)` — what is the first argument?**  
The first argument is width (ignored for vertical spacers). The second is height. It's a historical artifact — just always use `Spacer(1, desired_height)`.

**Key insight — `table` is reusable:** Once you've created a `Table` object, you can append it to multiple stories. But once `doc.build(story)` runs, the flowables in `story` are consumed. Create a new story list — don't reuse the old one.

---

In [ ]:
from reportlab.platypus import PageBreak                                                                                                                                                                          

  # ── CONFIDENTIAL header — runs on every page ──────────────────────────────────
def add_confidential_header(canvas, doc):
      canvas.saveState()
      canvas.setFont('Helvetica-Bold', 10)
      canvas.setFillColor(colors.red)
      canvas.drawCentredString(letter[0] / 2, letter[1] - 30, "CONFIDENTIAL")
      canvas.restoreState()

  # ── Start a new story — reuse styles/imports from the cell above ──────────────
story2 = []
story2.append(Paragraph("Compliance Evidence Report", title_style))
story2.append(Spacer(1, 0.2*inch))
story2.append(Paragraph(f"<b>Generated:</b> {datetime.utcnow().isoformat()}Z", styles['Normal']))
story2.append(Spacer(1, 0.3*inch))
story2.append(table)  # reuse the summary table from above

  # ── PageBreak: everything after this goes to page 2 ──────────────────────────
story2.append(PageBreak())

  # ── Page 2: Findings table ────────────────────────────────────────────────────
  story2.append(Paragraph("Detailed Findings", styles['Heading1']))
  story2.append(Spacer(1, 0.2*inch))

findings_data = [
      ['Finding ID',  'Severity',  'Description'],
      ['sh-001',      'CRITICAL',  'Root access key exists — delete immediately'],
      ['sh-002',      'HIGH',      'S3 bucket does not require SSL'],
      ['cfg-001',     'HIGH',      'IAM users missing MFA (3 of 7 users)'],
      ['cfg-002',     'MEDIUM',    'Password policy does not meet NIST requirements'],
      ['cfg-003',     'LOW',       'CloudWatch log group not encrypted'],
  ]

  findings_table = Table(findings_data, colWidths=[1.2*inch, 1*inch, 4.3*inch])
  findings_table.setStyle(TableStyle([
      ('BACKGROUND',  (0, 0), (-1, 0), colors.HexColor('#003366')),
      ('TEXTCOLOR',   (0, 0), (-1, 0), colors.whitesmoke),
      ('FONTNAME',    (0, 0), (-1, 0), 'Helvetica-Bold'),
      ('ALIGN',       (0, 0), (1, -1), 'CENTER'),
      ('ALIGN',       (2, 0), (2, -1), 'LEFT'),
      ('GRID',        (0, 0), (-1, -1), 1, colors.grey),
      ('BACKGROUND',  (0, 1), (-1, 1), colors.HexColor('#ffe0e0')),  # CRITICAL
      ('BACKGROUND',  (0, 2), (-1, 3), colors.HexColor('#fff0e0')),  # HIGH
      ('BACKGROUND',  (0, 4), (-1, 4), colors.HexColor('#fffde0')),  # MEDIUM
      ('BACKGROUND',  (0, 5), (-1, 5), colors.HexColor('#f0fff0')),  # LOW
  ]))
  story2.append(findings_table)

  # ── Build with CONFIDENTIAL header on every page ──────────────────────────────
  buffer2 = BytesIO()
  doc2 = SimpleDocTemplate(buffer2, pagesize=letter, topMargin=0.75*inch)
  doc2.build(story2,
             onFirstPage=add_confidential_header,
             onLaterPages=add_confidential_header)

  buffer2.seek(0)
  with open(r'C:\Users\jkl91\Downloads\extended_report.pdf', 'wb') as f:
      f.write(buffer2.getvalue())

  print(f"PDF saved to Downloads/extended_report.pdf")
  print(f"Size: {len(buffer2.getvalue()) / 1024:.1f} KB")
  print("Page 1: summary table + CONFIDENTIAL header")
  print("Page 2: findings table + CONFIDENTIAL header")

### Exercise 3.1: Extend the Hello World PDF

Modify the PDF above to add:
1. A `PageBreak()` after the summary table
2. A second table with sample findings (ID, Severity, Description)
3. A page header with "CONFIDENTIAL" text

**Hint:** Add `story.append(PageBreak())`, then build another `Table()` with finding data.

In [5]:
# Exercise 3.1 — builds directly on the cell above
# styles, title_style, table, colors, inch, letter, Table, TableStyle,
# Paragraph, Spacer, PageBreak, BytesIO, datetime are all already imported

# ── CONFIDENTIAL header function ──────────────────────────────────────────────
# This runs automatically every time ReportLab starts a new page.
# canvas = the raw drawing surface for that page (like a blank sheet of paper).
# saveState/restoreState = "don't let my red color leak into the rest of the doc"
def add_confidential_header(canvas, doc):
    canvas.saveState()
    canvas.setFont('Helvetica-Bold', 10)
    canvas.setFillColor(colors.red)
    canvas.drawCentredString(letter[0] / 2, letter[1] - 30, "CONFIDENTIAL")
    canvas.restoreState()

# ── Build a new story (can't reuse the old one — doc.build() already consumed it) ──
story2 = []

# Page 1: same title + timestamp + summary table as before
story2.append(Paragraph("Compliance Evidence Report", title_style))
story2.append(Spacer(1, 0.2 * inch))
story2.append(Paragraph(f"<b>Generated:</b> {datetime.utcnow().isoformat()}Z", styles['Normal']))
story2.append(Spacer(1, 0.3 * inch))
story2.append(table)  # reuse the summary table object from the cell above

# PageBreak — everything after this line goes onto page 2
story2.append(PageBreak())

# ── Page 2: Findings table ────────────────────────────────────────────────────
story2.append(Paragraph("Detailed Findings", styles['Heading1']))
story2.append(Spacer(1, 0.2 * inch))

# Each inner list = one row. First row = header.
findings_data = [
    ['Finding ID', 'Severity',  'Description'],
    ['sh-001',     'CRITICAL',  'Root access key exists — delete immediately'],
    ['sh-002',     'HIGH',      'S3 bucket does not require SSL'],
    ['cfg-001',    'HIGH',      'IAM users missing MFA (3 of 7 users)'],
    ['cfg-002',    'MEDIUM',    'Password policy does not meet NIST requirements'],
    ['cfg-003',    'LOW',       'CloudWatch log group not encrypted'],
]

findings_table = Table(findings_data, colWidths=[1.2 * inch, 1 * inch, 4.3 * inch])
findings_table.setStyle(TableStyle([
    # Header row: dark blue background, white text, bold
    ('BACKGROUND', (0, 0), (-1, 0),  colors.HexColor('#003366')),
    ('TEXTCOLOR',  (0, 0), (-1, 0),  colors.whitesmoke),
    ('FONTNAME',   (0, 0), (-1, 0),  'Helvetica-Bold'),
    # Center the ID and Severity columns; left-align Description
    ('ALIGN',      (0, 0), (1, -1),  'CENTER'),
    ('ALIGN',      (2, 0), (2, -1),  'LEFT'),
    # Grid lines
    ('GRID',       (0, 0), (-1, -1), 1, colors.grey),
    # Color-code each row by severity
    ('BACKGROUND', (0, 1), (-1, 1),  colors.HexColor('#ffe0e0')),  # CRITICAL = red tint
    ('BACKGROUND', (0, 2), (-1, 2),  colors.HexColor('#fff0e0')),  # HIGH
    ('BACKGROUND', (0, 3), (-1, 3),  colors.HexColor('#fff0e0')),  # HIGH
    ('BACKGROUND', (0, 4), (-1, 4),  colors.HexColor('#fffde0')),  # MEDIUM = yellow tint
    ('BACKGROUND', (0, 5), (-1, 5),  colors.HexColor('#f0fff0')),  # LOW = green tint
]))
story2.append(findings_table)

# ── Build the PDF with CONFIDENTIAL on every page ─────────────────────────────
# topMargin=0.75*inch leaves room at the top for the CONFIDENTIAL text
buffer2 = BytesIO()
doc2 = SimpleDocTemplate(buffer2, pagesize=letter, topMargin=0.75 * inch)
doc2.build(story2,
           onFirstPage=add_confidential_header,   # runs on page 1
           onLaterPages=add_confidential_header)  # runs on every page after

buffer2.seek(0)
with open(r'C:\Users\jkl91\Downloads\extended_report.pdf', 'wb') as f:
    f.write(buffer2.getvalue())

print(f"Saved to: C:\\Users\\jkl91\\Downloads\\extended_report.pdf")
print(f"Size: {len(buffer2.getvalue()) / 1024:.1f} KB")
print("Open it — you should see:")
print("  Page 1: CONFIDENTIAL header + summary table")
print("  Page 2: CONFIDENTIAL header + color-coded findings table")

Saved to: C:\Users\jkl91\Downloads\extended_report.pdf
Size: 3.2 KB
Open it — you should see:
  Page 1: CONFIDENTIAL header + summary table
  Page 2: CONFIDENTIAL header + color-coded findings table


C:\Users\jkl91\AppData\Local\Temp\ipykernel_27276\215837361.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  story2.append(Paragraph(f"<b>Generated:</b> {datetime.utcnow().isoformat()}Z", styles['Normal']))


---
### 🎨 Deep Dive: `canvas.saveState()` / `restoreState()` and Page Callbacks

**The canvas is a stateful drawing context.** Think of it like a painter's brush — whatever color you pick, it stays picked until you change it. `saveState()` takes a snapshot; `restoreState()` rewinds to that snapshot.

```python
def add_confidential_header(canvas, doc):
    canvas.saveState()           # push current state onto a stack
    canvas.setFont('Helvetica-Bold', 10)
    canvas.setFillColor(colors.red)
    canvas.drawCentredString(letter[0] / 2, letter[1] - 30, "CONFIDENTIAL")
    canvas.restoreState()        # pop back — red color is gone, font is reset
```

Without `saveState/restoreState`, your red color would bleed into every paragraph on the page. This is the same concept as:
- **Python context manager** (`with open(...) as f:`) — set up, do work, tear down
- **Database transaction** (`BEGIN / COMMIT / ROLLBACK`) — atomic state change
- **CSS `position: absolute`** inside a `position: relative` container — scoped coordinate system

**How `onFirstPage` / `onLaterPages` works:**

`doc.build(story, onFirstPage=fn, onLaterPages=fn)` wires a callback that ReportLab calls after laying out each page. The callback receives `(canvas, doc)` — you draw directly onto the finished page. This is why page numbers and headers work: by the time the callback fires, ReportLab knows the page dimensions and current page number (`doc.page`).

```python
def add_page_number(canvas, doc):
    canvas.saveState()
    canvas.setFont('Helvetica', 9)
    canvas.drawRightString(letter[0] - 0.5*inch, 0.5*inch, f"Page {doc.page}")
    canvas.restoreState()
```

**`letter[0] / 2`** — why index into `letter`? Because `letter` is a tuple: `(612.0, 792.0)`. Index 0 is width, index 1 is height. ReportLab measures in points (1 inch = 72 points). So `letter[1] - 30` is "30 points from the top of the page."

---

## Lab 3.2: Jinja2 Templating

Jinja2 separates report **structure** (template) from **data** (control findings). Non-technical users can tweak the template without touching Python. This is how production reporting systems work.

---
### 🏛️ Separation of Concerns: Data vs Presentation

This is the most important architectural principle in Lab 3.2. Jinja2 enforces it by design.

```
Python code                 Jinja2 template
──────────────              ───────────────────
computes the data           renders the data
owns business logic         owns layout decisions
tested with pytest          editable by non-engineers
```

This is **MVC** (Model-View-Controller) applied to reports:
- **Model** = `src/models.py` (ControlAssessment, CompliancePosture)
- **View** = Jinja2 template (what gets shown and how)
- **Controller** = `pdf_generator.py` (pulls data, passes to template)

**Why does this matter in practice?** A compliance analyst can update the template to add a new column or change formatting without touching any Python. The Python logic that calculates compliance percentages stays unchanged. This is the same reason Django uses `.html` templates instead of building HTML in Python strings.

**Jinja2 syntax cheat sheet (boilerplate you'll use every time):**

```jinja
{{ variable }}                       ← output a variable
{{ variable | filter }}              ← apply a filter
{{ dict.key }} or {{ dict['key'] }}  ← dict access (both work)
{% for item in list %}...{% endfor %} ← loop
{% if condition %}...{% elif %}...{% else %}...{% endif %}  ← conditional
{{ list | length }}                  ← filter: count items
{{ number | round(2) }}              ← filter: round to 2 decimal places
{{ controls | selectattr('status', 'equalto', 'PASS') | list | length }}  ← filter chain
```

**What Jinja2 cannot do** — and what you must do in Python before passing data:
- Sort a list by a field (do it with `sorted()` in Python first)
- Group items by a key (do it with a dict in Python first)
- Do math beyond simple arithmetic (compute it in Python, pass the result)

This is intentional: templates should be dumb display logic. If your template has complex logic, move it to Python.

---

In [4]:
!pip install jinja2 -q

from jinja2 import Template

template_string = """
{{ organization_name }} Compliance Report
Generated: {{ report_date }}

Overall Score: {{ compliance_pct }}%
Controls Assessed: {{ controls | length }}

{% for control in controls %}
{{ control.id }}: {{ control.name }}
  Status: {{ control.status }}
  {% if control.findings %}Findings: {{ control.findings }}{% endif %}
{% endfor %}
"""

template = Template(template_string)
output = template.render(
    organization_name='Acme Corporation',
    report_date='2026-05-03T15:30:00Z',
    compliance_pct=45.83,
    controls=[
        {'id': 'AC-2', 'name': 'Account Management', 'status': 'FAIL', 'findings': 2},
        {'id': 'AU-2', 'name': 'Event Logging', 'status': 'PASS', 'findings': 0},
        {'id': 'SC-7', 'name': 'Boundary Protection', 'status': 'PARTIAL', 'findings': 1},
    ]
)
print(output)
print("\nJinja2 key features: {{ var }}, {% for %}, {% if %}, filters (| length)")


Acme Corporation Compliance Report
Generated: 2026-05-03T15:30:00Z

Overall Score: 45.83%
Controls Assessed: 3


AC-2: Account Management
  Status: FAIL
  Findings: 2

AU-2: Event Logging
  Status: PASS
  

SC-7: Boundary Protection
  Status: PARTIAL
  Findings: 1


Jinja2 key features: {{ var }}, {% for %}, {% if %}, filters (| length)



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Exercise 3.2: Build a Jinja2 compliance template

Create a Jinja2 template that:
1. Color-codes status (use `{% if control.status == 'PASS' %}` blocks)
2. Groups controls by family using `{% for family in families %}`
3. Shows a severity breakdown per family

**Hint:** You'll need to pre-process the data into a nested dict: `{family: [controls...]}`

In [ ]:
#right now our data is flat and needs to be grouped by family
#we do this grouping in ptyhon before passing to jinja2, this template just loops it doesn't sort or group

 # Raw controls — flat list (what you'd get from assessments in the real pipeline)
controls = [
    {'id': 'AU-12', 'name': 'Audit Record Generation','family': 'AU', 'status': 'PASS',    'severity': 'NONE',     'findings': 0},
    {'id': 'SC-7',  'name': 'Boundary Protection',    'family': 'SC', 'status': 'PARTIAL', 'severity': 'HIGH',     'findings': 1},
    {'id': 'SC-8',  'name': 'Transmission Conf.',     'family': 'SC', 'status': 'FAIL',    'severity': 'HIGH',     'findings': 1},
    {'id': 'IA-2',  'name': 'Authentication',         'family': 'IA', 'status': 'FAIL',    'severity': 'HIGH',     'findings': 3},
  ]

 # Group into {family: [controls...]}
  # This is plain Python — nothing Jinja2 specific yet
families = {}
for control in controls:
    family = control['family']
    if family not in families:
        families[family] = []        # first time seeing this family — create empty list
    families[family].append(control) # add control to its family bucket

    #check on what we built
for family, items in families.items():
    print(f"{family}: {[c['id'] for c in items]}")


#before we touch jinja2 compute what we'll show counts of each servirty per family:

#severity breakdown per family
# Result: {'AC': {'CRITICAL': 1, 'HIGH': 1}, 'AU': {'NONE': 2}, ...}

severity_breakdown = {} #this is writing to a new dict, a dict can be a value in another dict, we can nest them, 
for family, items in families.items(): #by saying for family, items in families.items() are looping throug the dict we just built, 
    #the keys of that dict are family names, the values are lists of controls in that family, so we are unpacking those into family and item variables
    counts = {}
    for control in items: # we are looping through the controls in each family, and counting how many of each severity we have
        sev = control['severity'] #now that I have defined control in the loop, I can access its severity and store it in a variable 
        counts[sev] = counts.get(sev,0) +1 # this is defiining counts[sev] as the current count of that severity plus one
    severity_breakdown[family] = counts #now we create a variable in the severity breakdown dict for this family, and set it equal it will come out to look like {'AC': {'CRITICAL': 1, 'HIGH': 1}, 'AU': {'NONE': 2}, ...}

for family, counts in severity_breakdown.items():
    print(f"{family}: {counts}")


b

AU: ['AU-12']
SC: ['SC-7', 'SC-8']
IA: ['IA-2']
AU: {'NONE': 1}
SC: {'HIGH': 2}
IA: {'HIGH': 1}


---
### 🐍 Python Deep Dive: The Groupby Pattern

The grouping code above is one of the most reusable patterns in Python. You'll use it everywhere. Let's make it explicit:

**What we just built:**
```python
# Pattern: flat list → grouped dict
families = {}
for control in controls:
    family = control['family']          # get the grouping key
    if family not in families:
        families[family] = []           # first time: initialize empty bucket
    families[family].append(control)   # add to the right bucket
```

**Cleaner versions of the same pattern:**
```python
# Option 1: setdefault (most readable)
families = {}
for control in controls:
    families.setdefault(control['family'], []).append(control)

# Option 2: defaultdict (most Pythonic)
from collections import defaultdict
families = defaultdict(list)
for control in controls:
    families[control['family']].append(control)

# Option 3: itertools.groupby (requires pre-sorted data)
from itertools import groupby
sorted_controls = sorted(controls, key=lambda c: c['family'])
families = {k: list(v) for k, v in groupby(sorted_controls, key=lambda c: c['family'])}
```

**`counts.get(sev, 0) + 1`** — this is the frequency count pattern:
```python
counts = {}
counts[sev] = counts.get(sev, 0) + 1
# If 'sev' isn't in counts, .get() returns 0. Then we add 1 and store it.
# Same as: counts[sev] = counts[sev] + 1  but without KeyError on first access
```
Cleaner version: `from collections import Counter; Counter(c['severity'] for c in items)`

**Where this pattern appears in the real codebase:**
- `ControlMappingEngine.assess_all_controls()` in `src/mapper/engine.py` — groups evidence by `control_id`
- `ConfigCollector.RULE_TO_NIST_MAP` — maps rule names to lists of control IDs (inverted groupby)
- `CompliancePosture.by_family` — groups assessments by family code

**Database analogy:** This is `GROUP BY family_code` in SQL. The fact that we do it in Python (not SQL or DynamoDB) is because our data is already in memory from Phase 1/2. When data lives in DynamoDB, we'd use a GSI (Global Secondary Index) with `family_code` as the partition key to get the same grouping at query time.

---

In [8]:
#now we write a new template that for jinja2
from jinja2 import Template
# this is a jinja2 template, it looks like a normal string but we can use jinja2 syntax to loop and insert variables
#plain text "color coding = symbols and labels (pass = ✓, fail = ✗, partial = ~)"

template = Template("""
============================================================
  COMPLIANCE REPORT
  ============================================================

  {% for family, items in families.items() %}
  ------------------------------------------------------------
  FAMILY: {{ family }} ({{ items | length }} controls)
  Severity breakdown: {{ severity_breakdown[family] }}
  ------------------------------------------------------------
  {% for control in items %}
    {% if control.status == 'PASS' %}
    [✓ PASS    ] {{ control.id }} - {{ control.name }}
    {% elif control.status == 'FAIL' %}
    [✗ FAIL    ] {{ control.id }} - {{ control.name }}
                 Findings: {{ control.findings }} | Severity: {{ control.severity }}
    {% else %}
    [~ PARTIAL ] {{ control.id }} - {{ control.name }}
                 Findings: {{ control.findings }} | Severity: {{ control.severity }}
    {% endif %}
  {% endfor %}

  {% endfor %}
  ============================================================
  SUMMARY
  Total controls: {{ controls | length }}
  Passing: {{ controls | selectattr('status', 'equalto', 'PASS') | list | length }}
  Failing: {{ controls | selectattr('status', 'equalto', 'FAIL') | list | length }}
  Partial: {{ controls | selectattr('status', 'equalto', 'PARTIAL') | list | length }}
  ============================================================
  """)

output = template.render(
    families=families, 
    severity_breakdown=severity_breakdown,
    controls=controls
)
# what did I just do? I created a new template that loops through the families and controls, and uses if statements to show different symbols and labels based on the control status. 
# I also added a severity breakdown and a summary at the end. Then I rendered the template with the data we computed earlier.

print(output)



  COMPLIANCE REPORT

  
  ------------------------------------------------------------
  FAMILY: AU (1 controls)
  Severity breakdown: {'NONE': 1}
  ------------------------------------------------------------
  
    
    [✓ PASS    ] AU-12 - Audit Record Generation
    
  

  
  ------------------------------------------------------------
  FAMILY: SC (2 controls)
  Severity breakdown: {'HIGH': 2}
  ------------------------------------------------------------
  
    
    [~ PARTIAL ] SC-7 - Boundary Protection
                 Findings: 1 | Severity: HIGH
    
  
    
    [✗ FAIL    ] SC-8 - Transmission Conf.
                 Findings: 1 | Severity: HIGH
    
  

  
  ------------------------------------------------------------
  FAMILY: IA (1 controls)
  Severity breakdown: {'HIGH': 1}
  ------------------------------------------------------------
  
    
    [✗ FAIL    ] IA-2 - Authentication
                 Findings: 3 | Severity: HIGH
    
  

  
  SUMMARY
  Total controls: 4
 

---
### 🔗 Jinja2 Filter Pipeline — Read Left to Right, Understand Inside Out

The summary section of that template used filter chaining. This is the most confusing Jinja2 syntax for beginners:

```jinja
{{ controls | selectattr('status', 'equalto', 'PASS') | list | length }}
```

Read it as a Unix pipe: each `|` passes the result of the left side to the right side.

**Step by step:**
```
controls                                    ← Python list of dicts
  | selectattr('status', 'equalto', 'PASS') ← generator: yields only items where status=='PASS'
  | list                                    ← materialize the generator into a real list
  | length                                  ← count the items
```

**Why is `selectattr` a generator, not a list?** Because generators are lazy — they don't compute all results upfront. If you have 10,000 controls, `selectattr` doesn't create a 10,000-item filtered list in memory. It streams results one at a time. `| list` materializes it when you actually need it (here, to count).

**Equivalent Python:**
```python
# Jinja2:   controls | selectattr('status', 'equalto', 'PASS') | list | length
# Python:   len([c for c in controls if c['status'] == 'PASS'])

# Or with filter():
len(list(filter(lambda c: c['status'] == 'PASS', controls)))
```

**Common Jinja2 filters — memorize these:**
```jinja
{{ list | length }}                          ← len(list)
{{ list | first }}                           ← list[0]
{{ list | last }}                            ← list[-1]
{{ list | sort(attribute='severity') }}      ← sorted(list, key=lambda x: x['severity'])
{{ string | upper }}                         ← string.upper()
{{ string | truncate(50) }}                  ← string[:50] + '...'
{{ number | round(2) }}                      ← round(number, 2)
{{ list | selectattr('key', 'equalto', v) }} ← (x for x in list if x['key'] == v)
{{ list | rejectattr('key', 'equalto', v) }} ← (x for x in list if x['key'] != v)
```

**Edge case:** `selectattr` only works on lists of objects/dicts. If `controls` is empty, `selectattr` returns an empty generator — `| list` gives `[]`, `| length` gives `0`. No errors. This safe-by-default behavior is why template engines are preferred over `try/except` everywhere in view logic.

---

## Lab 3.3: Chaining from Phase 2 — Building Assessment Data

Now we connect to the real pipeline. Instead of creating mock classes, we import from `src/` and build the exact same evidence → scan → assessments → posture chain that Phase 2 taught us. The PDF generator consumes Phase 2's output.

**Important:** Zero inline class definitions. Everything comes from `src/`.

In [9]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

# Import canonical models — same classes as Phase 2
from src.models import (
    EvidenceItem, ScanResult, CollectorResult, ControlAssessment,
    ControlStatus, CompliancePosture, generate_scan_id,
    SEVERITY_WEIGHTS, CONTROL_FAMILIES
)
from src.mapper.control_catalog import NIST_CONTROL_CATALOG, get_control
from src.mapper.engine import ControlMappingEngine
from src.evidence.pdf_generator import PDFReportGenerator

print(f"Models loaded from src/")
print(f"  Control catalog: {len(NIST_CONTROL_CATALOG)} controls")
print(f"  PDFReportGenerator: {PDFReportGenerator.__module__}")
print(f"  ControlAssessment fields: {list(ControlAssessment.__dataclass_fields__.keys())[:6]}...")

Models loaded from src/
  Control catalog: 25 controls
  PDFReportGenerator: src.evidence.pdf_generator
  ControlAssessment fields: ['control_id', 'control_title', 'control_family', 'family_name', 'status', 'evidence']...


In [10]:
# Build evidence (same pattern as Phase 2 — review that notebook if this is unfamiliar)
evidence_items = [
    EvidenceItem(
        source="security_hub", finding_id="sh-iam4-001",
        title="IAM.4 Hardware MFA should be enabled for the root user",
        status="FAILED", severity="CRITICAL",
        resource_type="AWS::IAM::User",
        resource_id="arn:aws:iam::123456789012:root",
        timestamp="2024-01-15T10:00:00Z",
        remediation="Delete root access keys and enable hardware MFA",
        control_ids=["AC-2", "IA-2", "IA-2(1)"]
    ),
    EvidenceItem(
        source="security_hub", finding_id="sh-s3-005",
        title="S3.5 Buckets should require requests to use SSL",
        status="FAILED", severity="HIGH",
        resource_type="AWS::S3::Bucket",
        resource_id="arn:aws:s3:::production-data-bucket",
        timestamp="2024-01-15T10:01:00Z",
        remediation="Add bucket policy requiring aws:SecureTransport",
        control_ids=["SC-8", "SC-13"]
    ),
    EvidenceItem(
        source="config", finding_id="cfg-cloudtrail-001",
        title="multi-region-cloudtrail-enabled: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::CloudTrail::Trail",
        resource_id="arn:aws:cloudtrail:us-east-1:123456789012:trail/org-trail",
        timestamp="2024-01-15T10:02:00Z",
        control_ids=["AU-2", "AU-3", "AU-12"]
    ),
    EvidenceItem(
        source="config", finding_id="cfg-ebs-001",
        title="encrypted-volumes: NON_COMPLIANT",
        status="FAILED", severity="HIGH",
        resource_type="AWS::EC2::Volume",
        resource_id="vol-0abc123def456789",
        timestamp="2024-01-15T10:03:00Z",
        remediation="Enable EBS encryption by default",
        control_ids=["SC-13", "SC-28"]
    ),
    EvidenceItem(
        source="iam", finding_id="iam-pwpolicy-001",
        title="Password policy does not meet NIST requirements",
        status="FAILED", severity="MEDIUM",
        resource_type="AWS::IAM::AccountPasswordPolicy",
        resource_id="arn:aws:iam::123456789012:account-password-policy",
        timestamp="2024-01-15T10:04:00Z",
        remediation="Set minimum 12 chars, require complexity, 90-day rotation",
        control_ids=["IA-5(1)"]
    ),
    EvidenceItem(
        source="config", finding_id="cfg-gd-001",
        title="guardduty-enabled-centralized: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::GuardDuty::Detector",
        resource_id="detector-us-east-1",
        timestamp="2024-01-15T10:05:00Z",
        control_ids=["SI-4"]
    ),
    EvidenceItem(
        source="config", finding_id="cfg-mfa-001",
        title="iam-user-mfa-enabled: NON_COMPLIANT (3 of 7 users)",
        status="FAILED", severity="HIGH",
        resource_type="AWS::IAM::User",
        resource_id="arn:aws:iam::123456789012:user/developer-jane",
        timestamp="2024-01-15T10:06:00Z",
        remediation="Enforce MFA for all IAM users with console access",
        control_ids=["IA-2(1)", "AC-2"]
    ),
    EvidenceItem(
        source="config", finding_id="cfg-ssh-001",
        title="restricted-ssh: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::EC2::SecurityGroup",
        resource_id="sg-0abc123def456789",
        timestamp="2024-01-15T10:07:00Z",
        control_ids=["SC-7", "CM-6"]
    ),
    EvidenceItem(
        source="config", finding_id="cfg-ssm-001",
        title="ec2-instance-managed-by-systems-manager: COMPLIANT",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::EC2::Instance",
        resource_id="i-0abc123def456789",
        timestamp="2024-01-15T10:08:00Z",
        control_ids=["CM-2", "CM-8", "SI-2"]
    ),
    EvidenceItem(
        source="security_hub", finding_id="sh-iam1-001",
        title="IAM.1 No full * administrative privileges",
        status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::IAM::Policy",
        resource_id="arn:aws:iam::123456789012:policy/DeveloperAccess",
        timestamp="2024-01-15T10:09:00Z",
        control_ids=["AC-6", "AC-3"]
    ),
]

# Build ScanResult → run ControlMappingEngine → get posture (Phase 2 pipeline)
scan = ScanResult(scan_id=generate_scan_id(), scan_start="2024-01-15T10:00:00Z",
                  account_id="123456789012", region="us-east-1")
cr = CollectorResult(source="mock_collectors", status="SUCCESS",
                     evidence_items=evidence_items, raw_findings_count=len(evidence_items))
scan.collector_results["mock"] = cr
scan.finalize()

engine = ControlMappingEngine()
assessments = engine.assess_all_controls(scan)
posture = engine.generate_posture(assessments)

print(f"Phase 2 pipeline complete:")
print(f"  Evidence items: {len(evidence_items)}")
print(f"  Assessments: {len(assessments)} controls")
print(f"  Compliance: {posture.compliance_percentage:.1f}%")
print(f"  PASS: {posture.passed}  FAIL: {posture.failed}  PARTIAL: {posture.partial}  N/A: {posture.not_assessed}")
print(f"\nTop 3 failures:")
for f in posture.top_failures[:3]:
    print(f"  {f['control_id']}: {f['control_title']} (priority {f['remediation_priority']})")

Phase 2 pipeline complete:
  Evidence items: 10
  Assessments: 25 controls
  Compliance: 44.0%
  PASS: 11  FAIL: 7  PARTIAL: 0  N/A: 7

Top 3 failures:
  AC-2: Account Management (priority 10)
  IA-2: Identification and Authentication (Organizational Users) (priority 10)
  IA-2(1): Multi-Factor Authentication to Privileged Accounts (priority 10)


---
### 🔄 Pattern Recognition: This IS the Pipeline — Connecting All Three Phases

Study this code closely. Every single line connects back to something you built in Phase 1 or Phase 2.

**`EvidenceItem.control_ids` is the JOIN key:**
```python
EvidenceItem(
    ...
    control_ids=["AC-2", "IA-2", "IA-2(1)"]  ← this single finding maps to 3 controls
)
```
In Phase 2, `ControlMappingEngine.assess_all_controls()` iterates over every `EvidenceItem` in the scan and groups them by `control_id`. This is a **many-to-many relationship**: one finding can map to multiple controls, and one control gets evidence from multiple findings. In SQL terms: `EvidenceItem` → `control_ids` array → `ControlAssessment` is like a junction table.

**`ScanResult.collector_results` is a dict of dicts:**
```python
scan.collector_results["mock"] = cr   # key = collector name, value = CollectorResult
```
In production: `{"security_hub": CollectorResult(...), "config": CollectorResult(...), "iam": CollectorResult(...)}`. The mapping engine iterates all values and calls `.evidence_items` on each. Adding a new collector (e.g., GuardDuty) just means adding another key — the engine doesn't change.

**`scan.finalize()` — what does it do?**  
It sets `scan_end` timestamp and computes derived counts. After `finalize()`, `scan.all_evidence` is a flat list of all `EvidenceItem` objects from all collectors. The mapping engine uses `scan.all_evidence`.

**`engine.assess_all_controls(scan)` produces one `ControlAssessment` per control in the catalog:**
```
NIST_CONTROL_CATALOG (25 controls)
  ↓  for each control:
  ↓    filter evidence_items where control_id in item.control_ids
  ↓    if no evidence → NOT_ASSESSED
  ↓    if any FAILED → FAIL
  ↓    if all PASSED → PASS
  ↓    else → PARTIAL
  ↓
List[ControlAssessment] (25 items)
```

**`posture.top_failures` is pre-sorted:** `generate_posture()` sorts failures by `remediation_priority` descending and takes the top 10. You don't need to sort them again in the PDF — they're already in the right order.

---

## Lab 3.3: Generating the PDF with `src/evidence/pdf_generator.py`

Now we use the **production** `PDFReportGenerator` from `src/evidence/pdf_generator.py`. This class accepts the same `ControlAssessment` and `CompliancePosture` objects we just built. No wrapper classes, no adapters — direct pipeline chaining.

The generator has two modes:
- **ReportLab mode** — Full PDF with styled tables, color-coded statuses, family sections
- **Text fallback** — Plain-text report when ReportLab isn't installed (useful in CI/CD)

**DDIA Connection (Ch. 10 — Batch Processing, p. 419):** The PDF is a *derived dataset*. We could delete every PDF and regenerate them from stored `ControlAssessment` objects. This is the same principle behind materialized views — the PDF is a read-optimized projection of the assessment data.

In [11]:
# Generate PDF using the REAL PDFReportGenerator from src/
generator = PDFReportGenerator(use_reportlab=True)

# The generator takes assessments + posture + output path
output_path = '/tmp/compliance_evidence_report.pdf'
result_path = generator.generate(assessments, posture, output_path)

file_size = os.path.getsize(result_path)
print(f"PDF generated: {result_path}")
print(f"Size: {file_size / 1024:.1f} KB")
print(f"Pages: Cover + Executive Summary + {len(posture.by_family)} family sections + findings")
print(f"\nGenerator signature:")
print(f"  generator.generate(")
print(f"      assessments: List[ControlAssessment],  # {len(assessments)} controls")
print(f"      posture: CompliancePosture,             # {posture.compliance_percentage:.1f}% compliance")
print(f"      output_path: str                        # writes PDF file")
print(f"  ) -> str  # returns path to generated file")

PDF generated: /tmp/compliance_evidence_report.pdf
Size: 12.6 KB
Pages: Cover + Executive Summary + 8 family sections + findings

Generator signature:
  generator.generate(
      assessments: List[ControlAssessment],  # 25 controls
      posture: CompliancePosture,             # 44.0% compliance
      output_path: str                        # writes PDF file
  ) -> str  # returns path to generated file


In [12]:
# Text fallback mode — useful for CI/CD where reportlab may not be installed
text_generator = PDFReportGenerator(use_reportlab=False)
text_path = '/tmp/compliance_evidence_report.txt'
text_generator.generate(assessments, posture, text_path)

# Read first 40 lines to see the format
with open(text_path, 'r') as f:
    lines = f.readlines()[:40]
    for line in lines:
        print(line, end='')

print(f"\n... ({len(open(text_path).readlines())} total lines)")

AWS COMPLIANCE REPORT

Scan ID: 2026-05-05T19-53-59Z_56004464
Report Date: 2026-05-05 19:54:25 UTC

--------------------------------------------------------------------------------
EXECUTIVE SUMMARY
--------------------------------------------------------------------------------
Overall Compliance Score: 44.0%
Total Controls: 25
Applicable Controls: 25
Passed: 11
Failed: 7
Partial: 0
Not Assessed: 7

--------------------------------------------------------------------------------
COMPLIANCE BY FAMILY
--------------------------------------------------------------------------------
AC: 2/3 passed (66.7%)
AU: 3/4 passed (75.0%)
CM: 3/4 passed (75.0%)
CP: 0/2 passed (0.0%)
IA: 0/4 passed (0.0%)
RA: 0/1 passed (0.0%)
SC: 1/4 passed (25.0%)
SI: 2/3 passed (66.7%)

--------------------------------------------------------------------------------
TOP FAILURES (HIGH PRIORITY)
--------------------------------------------------------------------------------
[Priority 10/10] AC-2: Account Managemen

---
### 🔀 Why Two Output Modes? Defensive Coding and the Dependency Injection Pattern

`PDFReportGenerator(use_reportlab=True/False)` is **dependency injection at the flag level**. The class behavior changes based on a flag passed at construction time — not hardcoded inside the class. This is the same pattern you already know from `ConfigCollector(client=None)`:

```python
# ConfigCollector: inject a mock client for testing
collector = ConfigCollector(client=mock_client)  # uses your mock
collector = ConfigCollector()                    # creates a real boto3 client

# PDFReportGenerator: inject output format for environment
generator = PDFReportGenerator(use_reportlab=True)   # real PDF (production)
generator = PDFReportGenerator(use_reportlab=False)  # plain text (CI/testing)
```

**When does the text fallback matter in real life?**

| Environment | Why text fallback? |
|---|---|
| GitHub Actions CI | ReportLab may not be in the test image, or you'd need a full Lambda-sized Docker container |
| Lambda cold start testing | Verifying business logic without the 15MB ReportLab download |
| Email digests | `compliance_report.txt` can be attached to an SNS email notification directly |
| Log aggregation | Text output can be streamed to CloudWatch Logs; PDFs cannot |
| Compliance diff | `diff old_report.txt new_report.txt` — instant change detection between scans |

**Edge case — ReportLab import failure:**  
`PDFReportGenerator` likely wraps the ReportLab import in a `try/except ImportError`. If the import fails (not installed, wrong architecture), `use_reportlab` is automatically set to `False`. This is graceful degradation — the tool still works, just produces text instead of PDF.

**System design implication (DDIA Ch. 1 — Reliability):** A system that degrades gracefully under dependency failure is more reliable than one that raises an uncaught exception. The text fallback is the "circuit breaker" for the ReportLab dependency.

---

### Exercise 3.3: Extend the PDF Generator

Open `src/evidence/pdf_generator.py` and study its structure. Then:

1. **Add a SHA-256 integrity hash** to the cover page. Compute the hash from the serialized assessment data (`json.dumps([a.to_dict() for a in assessments])`). Add it as a `Paragraph` on the cover page. This gives auditors a way to verify the report hasn't been tampered with.

2. **Add a "Data Collection Methodology" appendix** section. After the remediation guide, add a page that documents: which AWS services were scanned, the scan timestamp, the account ID, and the region. Pull this from `posture.scan_id` and the `ScanResult` metadata.

3. **Test the text fallback** by setting `use_reportlab=False` and verifying the text report contains the same information.

**Hint:** Look at `_generate_pdf()` and `_build_cover_page()` in the source. Add your hash computation before `doc.build(story)`. Use `import hashlib; hashlib.sha256(data.encode()).hexdigest()`.

---
### ⚡ Lambda Handler Patterns — Translating This Notebook to Production Code

The full pipeline code above IS the Lambda handler, minus the DynamoDB load. Study the production translation:

**Idempotency — the most important Lambda design principle:**
```python
def lambda_handler(event, context):
    scan_id = event['scan_id']
    
    # CHECK: has this scan already been processed?
    # If yes, return the existing S3 key (don't regenerate)
    # This prevents duplicate reports if SNS re-delivers the message
    existing = check_if_report_exists(scan_id)
    if existing:
        return {'statusCode': 200, 'body': {'s3_key': existing}}
    
    # PROCESS: generate and upload
    ...
```
Lambda + SNS is "at-least-once delivery" — SNS can deliver the same message twice. Your handler must be idempotent (running it twice produces the same result, not two PDFs). Use a DynamoDB `ConditionExpression` to check if the scan was already processed.

**Error handling pattern (DVA-C02 Domain 1 — Development):**
```python
def lambda_handler(event, context):
    scan_id = event.get('scan_id')
    if not scan_id:
        raise ValueError("Missing scan_id in event")  # don't catch this — let Lambda retry
    
    try:
        assessments = load_assessments(scan_id)      # DynamoDB
        pdf_path = generator.generate(assessments, posture, f'/tmp/{scan_id}.pdf')
        s3_key = uploader.upload_evidence(pdf_path, scan_id, assessments)
        return {'statusCode': 200, 'body': {'s3_key': s3_key}}
    except ClientError as e:
        if e.response['Error']['Code'] == 'ProvisionedThroughputExceededException':
            raise  # re-raise to trigger Lambda retry with exponential backoff
        logger.error(f"Unretryable error: {e}")
        raise  # always re-raise — don't swallow errors in Lambda
```

**Cold start optimization:**
```python
# Module-level (outside handler) — initialized ONCE per container, not per invocation
engine = ControlMappingEngine()          # cheap object, but saves 10ms per call
generator = PDFReportGenerator()         # same
uploader = ComplianceEvidenceUploader('compliance-evidence-prod')

def lambda_handler(event, context):
    scan_id = event['scan_id']
    ...  # use module-level objects
```
Module-level initialization is "warm container reuse" — Lambda may reuse the same container for multiple invocations. Objects initialized outside `lambda_handler` persist across invocations, saving cold-start time for expensive operations (boto3 client creation, DB connections).

---

## Part 3: DDIA Deep Dive — Batch Processing and Derived Data (Ch. 10)

### The PDF Generation Pipeline as Batch Processing

Kleppmann defines batch processing as: *"processing a large volume of bounded data, producing derived datasets as output."* Our pipeline matches this exactly:

```
Immutable Input              Processing                    Derived Output
─────────────               ───────────                   ──────────────
ScanResult                  ControlMappingEngine          List[ControlAssessment]
 └─ all_evidence (10)        ├─ Group by control_id         └─ CompliancePosture
                             ├─ Assess each control              └─ PDFReportGenerator
                             └─ Calculate priority                     └─ PDF file (on S3)
```

**Three key DDIA principles we follow:**

1. **Immutable inputs** (p. 413): Neither the `ScanResult` nor the `ControlAssessment` list is modified by the PDF generator. It reads them and produces a new artifact. If PDF generation fails, we retry with the exact same inputs.

2. **Deterministic processing** (p. 422): Given the same `assessments` and `posture`, the PDF generator produces the same PDF content every time. (Timestamps on the cover page are the only variable, and those come from the input data, not `datetime.now()`.) This is critical for auditing — regenerating a report must produce identical findings.

3. **Derived datasets** (p. 419): The PDF is derived from `ControlAssessment` objects. We could delete every PDF in S3 and regenerate them from DynamoDB. This is the same idea as Kleppmann's materialized views: the PDF is a read-optimized projection that denormalizes assessment data into a human-readable format.

### Why Not Generate PDFs in Real Time?

**Batch vs stream trade-off (DDIA p. 464):** We generate PDFs on-demand (triggered by scan completion), not as a continuously updating stream. Why?

- PDFs are point-in-time snapshots — auditors want the state at a specific moment
- ReportLab rendering takes 1-3 seconds per report — too slow for real-time
- S3 gives us cheap, durable storage for historical reports
- Regeneration is cheap if we keep the assessment data

If we needed real-time compliance dashboards (Phase 5), we'd use a different output format (JSON API, not PDF). The PDF is the batch-optimized output; the API is the stream-optimized output.

---

---
### 🏗️ System Design: Where the PDF Fits in the Full Architecture

The DDIA section above is theory. Here's the concrete architecture this notebook is building toward:

```
Scan completion event
      │
      ▼
SNS Topic: scan-completed
      │
      ▼
Lambda: pdf-generator
  ├── load ScanResult from DynamoDB (PK: scan_id)
  ├── load ControlAssessments from DynamoDB (GSI: scan_id)
  ├── engine.generate_posture(assessments)
  ├── PDFReportGenerator.generate(assessments, posture, '/tmp/report.pdf')
  └── ComplianceEvidenceUploader.upload_evidence(pdf_path, scan_id, assessments)
          │
          ▼
         S3: compliance-evidence-prod/
              evidence/2024-01-15/{scan_id}/compliance-report.pdf
          │
          ▼
      (presigned URL → auditor email)
```

**Why Lambda, not a long-running server?**  
PDF generation is bursty — it happens once per scan, which might run daily or on-demand. You'd pay for a server running idle 23+ hours a day. Lambda runs for 1-3 seconds, bills per 100ms, then stops. For a 25-control report, Lambda is ~$0.0001 per report.

**Why SNS between scan completion and PDF generation?**  
Decoupling (DDIA Ch. 11). The scanner Lambda doesn't need to know that a PDF generator exists. It publishes "scan completed" to SNS; anyone who cares subscribes. Tomorrow if you add a Slack notification or a DynamoDB indexer, you just add another subscriber — the scanner never changes.

**DynamoDB access pattern for Phase 4:**
```
Table: compliance-evidence
  PK: scan_id            (e.g., "2026-05-05T19-53-59Z_56004464")
  SK: control_id         (e.g., "AC-2")
  attributes: status, severity, findings, priority, ...

GSI: by-control
  PK: control_id         ← query all scans for a given control
  SK: scan_id            ← sorted by time
```
This lets you ask: "Show me the AC-2 status across the last 30 scans" — the Phase 4 drift detection query.

---

## Lab 3.4: Lambda Layers for Dependencies

### DVA-C02 Connection (Domain 2 — Development)

ReportLab is ~15MB. Jinja2 is ~5MB. Bundling these into every Lambda zip is wasteful. Lambda Layers solve this.

**Layer structure:**
```
pdf-generator-layer.zip
└── python/lib/python3.11/site-packages/
    ├── reportlab/
    ├── jinja2/
    └── ...
```

Lambda loads up to 5 layers (250MB total). Dependencies live in layers; function code stays small and fast to deploy.

**Exam relevance:** Lambda Layers appear frequently on DVA-C02 Domain 2. Key facts:
- Max 5 layers per function
- Total unzipped size (function + layers) must be < 250MB
- Layers are versioned — you reference a specific ARN with version
- Layers are region-specific — publish in each region you deploy to

---
### 📦 Lambda Size Budget — A Practical Decision Framework

Lambda layers exist because Lambda has hard size limits. Knowing the budget is an exam topic and a real engineering constraint.

**Size limits (memorize for DVA-C02):**
```
Lambda function zip (compressed):     50 MB
Lambda function zip (unzipped):       250 MB  ← total including all layers
Single layer (unzipped):              250 MB
Max layers per function:              5
/tmp storage (for PDF writing):       512 MB default, up to 10 GB configurable
Memory (affects CPU speed too):       128 MB – 10,240 MB
Max execution time:                   15 minutes
```

**Our compliance tool's approximate footprint:**
```
reportlab (unzipped):    ~15 MB
jinja2 (unzipped):       ~5 MB
boto3 (pre-installed):   0 MB  ← AWS provides this automatically in Lambda runtime
botocore:                0 MB  ← same
src/ package:            ~1 MB
Total:                   ~21 MB  ← well within 250MB limit
```

**What happens if you exceed 250MB?** The `CreateFunction` or `UpdateFunctionCode` API call fails with `CodeStorageExceededException`. Common cause: bundling pandas, numpy, or scipy (each ~50-100MB). Solution: use Lambda container images (up to 10GB) instead of zip layers.

**The `/tmp` directory on Lambda:**  
Lambda's filesystem is read-only except for `/tmp`. That's why `PDFReportGenerator` writes to `/tmp/report.pdf`. In the lab notebook on Windows we changed this to `C:\Users\jkl91\Downloads\` because `/tmp` doesn't exist on Windows — but the production code always uses `/tmp` in Lambda.

**Edge case — Lambda concurrency and `/tmp`:**  
Multiple Lambda invocations can run simultaneously on different containers. Each container has its own `/tmp` — they don't share. But if Lambda reuses a warm container (rare on Lambda but common on Fargate), `/tmp` from a previous invocation may still be there. Always use unique filenames: `f'/tmp/report_{scan_id}.pdf'`.

---

In [13]:
import zipfile, shutil

# Simulate creating a Lambda layer (in production, you'd install packages here)
layer_dir = '/tmp/pdf-generator-layer'
site_packages = f'{layer_dir}/python/lib/python3.11/site-packages'
os.makedirs(site_packages, exist_ok=True)

# In real build: pip install reportlab jinja2 -t site_packages
# For demo, we create a placeholder
with open(f'{site_packages}/__init__.py', 'w') as f:
    f.write("# PDF generator layer placeholder\n")

# Create the zip
layer_zip = '/tmp/pdf-generator-layer.zip'
with zipfile.ZipFile(layer_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(layer_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, layer_dir)
            zf.write(file_path, arcname)

print(f"Layer zip: {os.path.getsize(layer_zip)} bytes")
print(f"\nIn production, this is how you publish it:")
print("""
# Build layer (run on Amazon Linux 2 or in Docker):
docker run --rm -v $(pwd):/out python:3.11 bash -c \\
  'pip install reportlab jinja2 -t /out/python/lib/python3.11/site-packages'
zip -r pdf-generator-layer.zip python/

# Terraform (from terraform/modules/lambda/main.tf):
resource "aws_lambda_layer_version" "pdf_generator" {
  filename   = "artifacts/pdf-generator-layer.zip"
  layer_name = "pdf-generator-layer"
  compatible_runtimes = ["python3.11", "python3.12"]
}

resource "aws_lambda_function" "evidence_generator" {
  layers = [aws_lambda_layer_version.pdf_generator.arn]
  ...
}
""")

# Clean up
shutil.rmtree(layer_dir)
os.remove(layer_zip)

Layer zip: 229 bytes

In production, this is how you publish it:

# Build layer (run on Amazon Linux 2 or in Docker):
docker run --rm -v $(pwd):/out python:3.11 bash -c \
  'pip install reportlab jinja2 -t /out/python/lib/python3.11/site-packages'
zip -r pdf-generator-layer.zip python/

# Terraform (from terraform/modules/lambda/main.tf):
resource "aws_lambda_layer_version" "pdf_generator" {
  filename   = "artifacts/pdf-generator-layer.zip"
  layer_name = "pdf-generator-layer"
  compatible_runtimes = ["python3.11", "python3.12"]
}

resource "aws_lambda_function" "evidence_generator" {
  layers = [aws_lambda_layer_version.pdf_generator.arn]
  ...
}



## Lab 3.5: S3 Integration and Presigned URLs

### DVA-C02 Connection (Domain 2 & 3 — Development and Security)

Once we generate a PDF, we store it in S3 and create **presigned URLs** so auditors can download evidence without AWS credentials. Links expire after 24 hours.

**Exam pattern:** "An application generates compliance reports and needs to share them securely with external auditors who don't have AWS accounts. Which approach is most secure?" → Presigned URLs with short expiration.

**Key S3 concepts for the exam:**
- `ServerSideEncryption='AES256'` — Encrypt at rest (SSE-S3)
- `StorageClass='STANDARD_IA'` — Cheaper for infrequent access (audit reports)
- Lifecycle policies: transition to Glacier after 90 days, Deep Archive after 180 days, delete after 7 years

---
### 🔒 DVA-C02 Quick Reference: S3 Security Patterns

This section is pure exam prep. These patterns appear constantly on the DVA-C02.

**Encryption at rest — three modes:**
```
SSE-S3   (AES256):     AWS manages the key. Simplest. No extra cost.
                       Header: ServerSideEncryption='AES256'

SSE-KMS  (aws:kms):    Customer-managed key in KMS. Audit trail per-access.
                       FedRAMP requires this for MODERATE/HIGH data.
                       Header: ServerSideEncryption='aws:kms', SSEKMSKeyId='arn:...'

SSE-C    (customer):   You provide the key with every request. Rare. Painful.
                       AWS doesn't store the key — you lose it, you lose the data.
```

**Presigned URLs — how they work:**
```python
# You (with s3:GetObject permission) generate a URL
url = s3.generate_presigned_url('get_object',
    Params={'Bucket': 'my-bucket', 'Key': 'report.pdf'},
    ExpiresIn=86400   # 24 hours in seconds
)
# The URL embeds: your credentials (temporary), expiration, bucket, key, signature
# Anyone with the URL can GET the object until it expires — no AWS account needed
```

**Exam gotchas on presigned URLs:**
- The URL uses the **generating principal's permissions**, not the downloader's
- If the generating IAM role is deleted, existing presigned URLs stop working immediately
- Presigned URLs can also be used for `PUT` (allowing external upload to S3)
- Max expiration: 7 days (604800 seconds) for IAM role credentials; 12 hours for root or EC2 metadata

**S3 Access Control layers (evaluated in order):**
```
1. Block Public Access (account/bucket level) — overrides everything
2. Bucket Policy (JSON on the bucket)
3. IAM Policy (on the principal calling the API)
4. ACL (legacy — avoid for new buckets)
```

**Storage classes and when to use each:**
```
STANDARD          → frequent access (<30 days expected)
STANDARD_IA       → infrequent access, but needs millisecond retrieval (compliance reports)
ONE_ZONE_IA       → same as IA but single AZ — cheaper, less durable
GLACIER Instant   → archived, retrieved in milliseconds
GLACIER Flexible  → archived, retrieved in 1-5 minutes
DEEP_ARCHIVE      → cheapest, 12-hour retrieval (7+ year retention)
```

---

In [14]:
import boto3
from botocore.exceptions import ClientError
import hashlib, json

class ComplianceEvidenceUploader:
    """Upload compliance PDFs to S3 and generate presigned URLs.
    
    Uses src/models types directly — ControlAssessment.to_dict() for
    serialization, posture.scan_id for S3 key partitioning.
    """
    
    def __init__(self, bucket_name: str, region: str = 'us-east-1'):
        self.bucket_name = bucket_name
        self.s3_client = boto3.client('s3', region_name=region)
    
    def upload_evidence(self, pdf_path: str, scan_id: str, 
                        assessments: list) -> str:
        """Upload PDF to S3 with audit metadata."""
        # S3 key uses date-based partitioning (DDIA Ch. 6)
        date_prefix = scan_id.split('T')[0] if 'T' in scan_id else scan_id[:10]
        key = f"evidence/{date_prefix}/{scan_id}/compliance-report.pdf"
        
        # Compute integrity hash from assessment data
        data_json = json.dumps([a.to_dict() for a in assessments], default=str)
        data_hash = hashlib.sha256(data_json.encode()).hexdigest()
        
        with open(pdf_path, 'rb') as f:
            self.s3_client.put_object(
                Bucket=self.bucket_name, Key=key, Body=f.read(),
                ContentType='application/pdf',
                ServerSideEncryption='AES256',
                StorageClass='STANDARD_IA',
                Metadata={
                    'scan-id': scan_id,
                    'data-hash-sha256': data_hash,
                    'generator': 'compliance-evidence-collector-v1.0',
                    'assessment-count': str(len(assessments)),
                }
            )
        return key
    
    def generate_presigned_url(self, key: str, expiration_hours: int = 24) -> str:
        """Generate time-limited download URL for auditors."""
        return self.s3_client.generate_presigned_url(
            'get_object',
            Params={'Bucket': self.bucket_name, 'Key': key},
            ExpiresIn=expiration_hours * 3600
        )

# Demo the API (no real AWS call — shows the interface)
print("ComplianceEvidenceUploader API:")
print(f"  upload_evidence(pdf_path, scan_id, assessments) -> S3 key")
print(f"  generate_presigned_url(key, expiration_hours=24) -> URL")
print(f"\nExample S3 key: evidence/2024-01-15/{scan.scan_id}/compliance-report.pdf")
print(f"\nIntegrity hash (from {len(assessments)} assessments):")
data_json = json.dumps([a.to_dict() for a in assessments], default=str)
print(f"  SHA-256: {hashlib.sha256(data_json.encode()).hexdigest()[:32]}...")

ComplianceEvidenceUploader API:
  upload_evidence(pdf_path, scan_id, assessments) -> S3 key
  generate_presigned_url(key, expiration_hours=24) -> URL

Example S3 key: evidence/2024-01-15/2026-05-05T19-53-59Z_56004464/compliance-report.pdf

Integrity hash (from 25 assessments):
  SHA-256: 961b42fd244ef79bdad642572e9ac210...


---
### 🗂️ S3 Key Design — Partitioning and the DDIA Ch. 6 Connection

Look at this line from the uploader:
```python
key = f"evidence/{date_prefix}/{scan_id}/compliance-report.pdf"
# Example: evidence/2024-01-15/2024-01-15T10-00-00Z_abc123/compliance-report.pdf
```

This is **prefix-based partitioning** (DDIA Ch. 6, p. 202). Every S3 key is a path, and the prefix determines how AWS distributes requests across storage partitions. Good key design matters for performance and organization.

**Why `date/scan_id/` instead of `scan_id/` directly?**
```
BAD:  evidence/{scan_id}/report.pdf          ← all scans in one flat namespace
GOOD: evidence/{date}/{scan_id}/report.pdf   ← organized by date, easy to lifecycle
```

Benefits of date-based prefix:
1. **Lifecycle policies can target a date range** — `evidence/2024-*` can transition to Glacier together
2. **Auditors browse by date** — S3 Console shows folders; `evidence/2024-01-15/` is intuitive
3. **Avoids S3 rate limiting on hot prefixes** — S3 partitions storage by prefix hash; spreading scans across dates prevents all requests hitting the same partition

**The `Metadata` dict — what it enables:**
```python
Metadata={
    'scan-id': scan_id,
    'data-hash-sha256': data_hash,  # ← integrity verification without downloading
    'generator': 'compliance-evidence-collector-v1.0',
    'assessment-count': str(len(assessments)),
}
```
You can read S3 metadata with `head_object()` — a fast, cheap call that returns headers without downloading the file body. An auditor's verification tool can call `head_object()` to check the hash, then `get_object()` only if needed. This is the same principle as HTTP `HEAD` requests vs `GET`.

**`hashlib.sha256(data_json.encode()).hexdigest()`** — why SHA-256?
- SHA-256 is NIST-approved (FIPS 180-4) — required for FedRAMP evidence integrity
- `data_json.encode()` converts the string to bytes (SHA-256 operates on bytes, not strings)
- `.hexdigest()` returns a 64-character hex string (more portable than `.digest()` bytes)
- Same algorithm used by git for commit hashes, AWS S3 ETags, and TLS certificates

---

### Exercise 3.4: S3 Lifecycle Policy for Compliance Retention

FedRAMP requires evidence retention for 7 years. Design an S3 lifecycle policy that:

1. Keeps reports in STANDARD_IA for 90 days (frequent auditor access)
2. Transitions to GLACIER after 90 days (cheaper, retrieval takes minutes)
3. Transitions to DEEP_ARCHIVE after 180 days (cheapest, retrieval takes hours)
4. Deletes after 7 years (2,555 days)

Write the `put_bucket_lifecycle_configuration()` call. Then add a `verify_integrity(key)` method to `ComplianceEvidenceUploader` that downloads a report and verifies its SHA-256 hash against the metadata.

**Hint:** Use `s3_client.head_object()` to read metadata without downloading the file body.

## Lab 3.6: Full Pipeline — Evidence → PDF → (S3)

Let's run the complete Phase 3 pipeline end-to-end. This is the code that would run in a Lambda function triggered by scan completion.

In [ ]:
# Full pipeline: Evidence → Scan → Assessments → Posture → PDF
# This is what the Lambda handler does in production

from datetime import datetime, timezone

# Step 1: Build scan (Phase 1 output — in production, this comes from DynamoDB)
scan = ScanResult(
    scan_id=generate_scan_id(),
    scan_start=datetime.now(timezone.utc).isoformat(),
    account_id="123456789012", region="us-east-1"
)
cr = CollectorResult(source="all_collectors", status="SUCCESS",
                     evidence_items=evidence_items, 
                     raw_findings_count=len(evidence_items))
scan.collector_results["all"] = cr
scan.finalize()

# Step 2: Map to controls (Phase 2)
engine = ControlMappingEngine()
assessments = engine.assess_all_controls(scan)
posture = engine.generate_posture(assessments)

# Step 3: Generate PDF (Phase 3 — THIS phase)
generator = PDFReportGenerator(use_reportlab=True)
pdf_path = f'/tmp/evidence_{scan.scan_id}.pdf'
generator.generate(assessments, posture, pdf_path)

# Step 4: In production, upload to S3
# uploader = ComplianceEvidenceUploader('compliance-evidence-prod')
# s3_key = uploader.upload_evidence(pdf_path, scan.scan_id, assessments)
# url = uploader.generate_presigned_url(s3_key, expiration_hours=24)

print("=== PIPELINE COMPLETE ===")
print(f"Scan: {scan.scan_id}")
print(f"Evidence: {len(scan.all_evidence)} items from {len(scan.collector_results)} collector(s)")
print(f"Assessments: {len(assessments)} controls")
print(f"Compliance: {posture.compliance_percentage:.1f}%")
print(f"PDF: {pdf_path} ({os.path.getsize(pdf_path) / 1024:.1f} KB)")
print(f"\nLambda handler signature:")
print("""
def lambda_handler(event, context):
    scan_id = event['scan_id']
    scan = load_scan_from_dynamodb(scan_id)           # Phase 1 data
    assessments = engine.assess_all_controls(scan)     # Phase 2
    posture = engine.generate_posture(assessments)     # Phase 2
    pdf_path = generator.generate(assessments, posture, '/tmp/report.pdf')  # Phase 3
    s3_key = uploader.upload_evidence(pdf_path, scan_id, assessments)       # Phase 3
    return {'statusCode': 200, 'body': {'s3_key': s3_key}}
""")

## Exercises

### Exercise 3.5: Write a Lambda Handler

Write a complete `lambda_handler` function that:
1. Receives `{'scan_id': '...'}` from an SNS trigger (scan completion)
2. Loads `ControlAssessment` items from DynamoDB using `ControlAssessment.from_dict()`
3. Reconstructs the `CompliancePosture` using `engine.generate_posture()`
4. Generates a PDF and uploads to S3
5. Returns the presigned URL

Use `src/models.ControlAssessment.from_dict()` for deserialization. Think about error handling: what if DynamoDB is throttled? What if S3 upload fails? (DDIA Ch. 10 — fault tolerance in batch jobs)

### Exercise 3.6: Compare PDF vs Text Output

Generate both PDF and text reports for the same assessment data. Compare:
1. File size (PDF vs text)
2. Information content — do both contain the same data?
3. Which would an auditor prefer, and why?
4. When would you choose text over PDF? (Hint: CI/CD pipelines, email digests)

### Exercise 3.7: Schema Evolution Challenge (DDIA Ch. 4)

Our `ControlAssessment` has a `fedramp_baseline` field (string). Suppose we need to change it to `fedramp_baselines` (list of strings) because a control can be in multiple baselines.

1. What happens to existing PDFs that were generated with the old schema?
2. What happens to the PDF generator if we change the field? Does it break?
3. How would you handle backward compatibility? (Hint: read DDIA p. 112 on schema evolution with Avro)
4. Write a migration function that converts old `ControlAssessment` dicts to the new format.

### Exercise 3.8: Performance Benchmark

Generate a PDF with all 24 controls having 5+ evidence items each. Measure:
1. `time.time()` before and after `generator.generate()`
2. Memory usage with `tracemalloc`
3. PDF file size

What's the bottleneck? (Hint: ReportLab table rendering is O(rows × columns))

---

## Summary

### What you built
- Generated audit-ready PDFs using `src/evidence/pdf_generator.PDFReportGenerator`
- Chained Phase 2 output (assessments + posture) directly into PDF generation
- Explored ReportLab flowables, tables, and styling
- Built an S3 uploader with presigned URLs and integrity hashing
- Packaged dependencies as Lambda layers

### Pipeline so far
```
Phase 1: Collectors → ScanResult (evidence from AWS)
Phase 2: ScanResult → ControlMappingEngine → List[ControlAssessment] + CompliancePosture
Phase 3: ControlAssessment + CompliancePosture → PDFReportGenerator → PDF file → S3
Phase 4: (next) → Compare two scans → DriftEvent[] → SNS alerts
```

### Key classes used (all from `src/`)
| Class | Module | Role |
|-------|--------|------|
| `EvidenceItem` | `src.models` | Atomic unit of evidence |
| `ScanResult` | `src.models` | Phase 1 output |
| `ControlMappingEngine` | `src.mapper.engine` | Phase 2 processor |
| `ControlAssessment` | `src.models` | Per-control assessment |
| `CompliancePosture` | `src.models` | Aggregate summary |
| `PDFReportGenerator` | `src.evidence.pdf_generator` | **This phase** |

### DDIA connections
- **Ch. 10 (Batch Processing):** PDF is a derived dataset from immutable assessment data
- **Ch. 4 (Encoding and Evolution):** Schema changes in ControlAssessment affect PDF output
- **Ch. 6 (Partitioning):** S3 key structure partitions reports by date and scan ID

### DVA-C02 connections
- **Lambda Layers:** Package ReportLab/Jinja2 as reusable layers (Domain 2)
- **S3 Presigned URLs:** Secure evidence sharing without IAM credentials (Domain 3)
- **S3 Lifecycle Policies:** 7-year retention for FedRAMP compliance (Domain 2)
- **S3 Storage Classes:** STANDARD_IA → GLACIER → DEEP_ARCHIVE transition (Domain 2)

**Next: Phase 4** — Store assessments in DynamoDB, compare scans, detect compliance drift.